In [1]:
from sklearn.datasets import load_iris

x, y = load_iris(return_X_y=True)

print("x shape:", x.shape)
print("y shape:", y.shape)

x shape: (150, 4)
y shape: (150,)


In [2]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

base_models = [
    ("logistic", LogisticRegression(max_iter=1000)),
    ("tree", DecisionTreeClassifier(random_state=42)),
    ("knn", KNeighborsClassifier())
]

In [4]:
from sklearn.ensemble import StackingClassifier

stacking_model = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(max_iter=1000)
)

In [5]:
stacking_model.fit(
    x_train,
    y_train
)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.The type of estimator is generally expected to be a classifier.However, one can pass a regressor for some use case (e.g. ordinalregression).","[('logistic', ...), ('tree', ...), ...]"
,"final_estimator final_estimator: estimator, default=NoneA classifier which will be used to combine the base estimators.The default classifier is a:class:`~sklearn.linear_model.LogisticRegression`.",LogisticRegre...max_iter=1000)
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",None
,"stack_method stack_method: {'auto', 'predict_proba', 'decision_function', 'predict'}, default='auto'Methods called for each base estimator. It can be:* if 'auto', it will try to invoke, for each estimator, `'predict_proba'`, `'decision_function'` or `'predict'` in that order.* otherwise, one of `'predict_proba'`, `'decision_function'` or `'predict'`. If the method is not implemented by the estimator, it will raise an error.",'auto'
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary <n_jobs>` for more details.",None
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,) or list of ndarray if `y` is of type `""multilabel-indicator""`.Class labels.","ndarray[int64](3,)","[0,1,2]"
"estimators_ estimators_: list of estimatorsThe elements of the `estimators` parameter, having been fitted on thetraining data. If an estimator has been set to `'drop'`, itwill not appear in `estimators_`. When `cv=""prefit""`, `estimators_`is set to `estimators` and is not fitted again.",list,"[LogisticRegre...max_iter=1000), DecisionTreeC...ndom_state=42), KNeighborsClassifier()]"
final_estimator_ final_estimator_: estimatorThe classifier fit on the output of `estimators_` and responsible forfinal predictions.,LogisticRegression,LogisticRegre...max_iter=1000)


In [6]:
stacking_accuracy = stacking_model.score(
    x_test,
    y_test
)

print("Stacking Test Accuracy:", stacking_accuracy)

Stacking Test Accuracy: 1.0


In [7]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "Stacking": stacking_model
}

In [8]:
for name, model in models.items():

    model.fit(x_train, y_train)

    accuracy = model.score(
        x_test,
        y_test
    )

    print(
        f"{name:20} | "
        f"Test Accuracy = {accuracy:.4f}"
    )

Logistic Regression  | Test Accuracy = 0.9667
Decision Tree        | Test Accuracy = 0.9333
KNN                  | Test Accuracy = 1.0000
Stacking             | Test Accuracy = 1.0000


# 06. Stacking

## 1. What is Stacking?

**Stacking** is an ensemble learning technique where we combine the predictions of multiple different models and use another model, called a **meta-model**, to make the final prediction.

Instead of depending on only one model:

```text
x
↓
One Model
↓
Prediction
```

Stacking uses multiple models:

```text
                    x
                    ↓
        ┌───────────┼───────────┐
        ↓           ↓           ↓
       KNN         Tree         LR
        ↓           ↓           ↓
   Prediction  Prediction  Prediction
        └───────────┼───────────┘
                    ↓
              Meta-Model
                    ↓
             Final Prediction
```

---

# 2. Why do we use Stacking?

Different machine learning models can learn different patterns from the same data.

For example:

```text
KNN
→ Looks at nearby data points

Decision Tree
→ Learns decision rules

Logistic Regression
→ Learns a linear relationship between features and classes
```

Therefore, one model may make a mistake where another model makes the correct prediction.

Stacking tries to **combine their strengths**.

---

# 3. Base Models

The models at the first level are called **base models**.

In our experiment we used:

```text
Logistic Regression
Decision Tree
KNN
```

Conceptually:

```text
                    x
                    ↓
          ┌─────────┼─────────┐
          ↓         ↓         ↓
         LR        Tree       KNN
          ↓         ↓         ↓
          Prediction outputs
```

Each model makes its own prediction.

---

# 4. What is the Meta-Model?

The **meta-model** is the model that receives information from the base models and makes the final prediction.

```text
Base Model 1 ──┐
Base Model 2 ──┼──→ Meta-Model → Final Prediction
Base Model 3 ──┘
```

Think of it simply as:

```text
Base models
→ Give their opinions

Meta-model
→ Looks at those opinions
→ Makes the final decision
```

---

# 5. What is Fed to the Meta-Model?

The base models provide their **predictions** to the meta-model.

For example, suppose the three models predict:

```text
KNN       → Class 0
Decision Tree → Class 0
Logistic Regression → Class 1
```

Conceptually, the meta-model receives:

```text
[0, 0, 1]
```

For classification, stacking can also use **prediction probabilities**.

For example:

```text
KNN:
[0.90, 0.05, 0.05]

Decision Tree:
[0.70, 0.20, 0.10]

Logistic Regression:
[0.80, 0.15, 0.05]
```

The meta-model can use this information to learn how to combine the models' outputs.

The important idea is:

> **The meta-model learns from the predictions of the base models.**

---

# 6. Why do we need a Meta-Model?

Suppose three models give:

```text
KNN       → Cat
Tree      → Cat
Logistic  → Dog
```

One simple approach would be majority voting:

```text
Cat → 2 votes
Dog → 1 vote

Final → Cat
```

But stacking goes one step further.

Instead of manually deciding how to combine the models, we allow another model to **learn how to combine their predictions**.

```text
KNN prediction
Tree prediction
LR prediction
       ↓
   Meta-model
       ↓
Final prediction
```

So:

```text
Voting
→ Combine predictions using a voting rule

Stacking
→ Learn how to combine predictions using a meta-model
```

---

# 7. Does the Meta-Model Have to Be One of the Base Models?

**No.**

The meta-model can be a different algorithm.

For example:

```text
Base Models:
→ KNN
→ Decision Tree
→ Logistic Regression

Meta-Model:
→ Random Forest
```

This is completely valid.

Or:

```text
Base Models:
→ KNN
→ Decision Tree
→ Random Forest

Meta-Model:
→ Logistic Regression
```

Also valid.

There is no rule saying:

```text
Base Model = Meta Model
```

---

# 8. Why Did We Use Logistic Regression as the Meta-Model?

We used:

```python
final_estimator=LogisticRegression(max_iter=1000)
```

mainly because it is a **simple model that can learn how to combine the outputs of the base models**.

It is easy to understand and works well as a meta-model for this learning example.

It does **not** mean:

```text
Stacking → Always use Logistic Regression
```

Instead remember:

```text
Stacking
→ Base models
→ Meta-model
```

The meta-model can be chosen according to the problem.

---

# 9. Creating the Base Models

```python
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

base_models = [
    ("logistic", LogisticRegression(max_iter=1000)),
    ("tree", DecisionTreeClassifier(random_state=42)),
    ("knn", KNeighborsClassifier())
]
```

Here we create three base models:

```text
Logistic Regression
Decision Tree
KNN
```

The names:

```python
"logistic"
"tree"
"knn"
```

are simply labels used to identify the models.

---

# 10. Creating the Stacking Classifier

```python
from sklearn.ensemble import StackingClassifier

stacking_model = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(max_iter=1000)
)
```

### `estimators`

```python
estimators=base_models
```

specifies our **base models**.

```text
LR
Tree
KNN
```

### `final_estimator`

```python
final_estimator=LogisticRegression(...)
```

specifies our **meta-model**.

So the complete structure is:

```text
Base Models
    ↓
LR + Tree + KNN
    ↓
Their predictions
    ↓
Meta-model
    ↓
Logistic Regression
    ↓
Final prediction
```

---

# 11. Training the Stacking Model

```python
stacking_model.fit(
    x_train,
    y_train
)
```

We give the stacking system the training data.

Conceptually:

```text
x_train
   ↓
┌──────┬──────┬──────┐
│  LR  │ Tree │ KNN  │
└──────┴──────┴──────┘
   ↓      ↓      ↓
Predictions
   ↓      ↓      ↓
┌──────────────────┐
│    Meta-model    │
└────────┬─────────┘
         ↓
   Final prediction
```

The important point is that the meta-model learns how to use the outputs of the base models.

`StackingClassifier` handles the internal training process for us.

---

# 12. Evaluating the Stacking Model

```python
stacking_accuracy = stacking_model.score(
    x_test,
    y_test
)

print("Stacking Test Accuracy:", stacking_accuracy)
```

Here:

```text
x_test
→ unseen input data

y_test
→ actual answers
```

The complete stacking system makes predictions and compares them with `y_test`.

---

# 13. Our Stacking Result

We got:

```text
Stacking Test Accuracy: 1.0
```

Therefore:

```text
100% Test Accuracy
```

Our test set contained:

```text
30 samples
```

So:

```text
30 / 30 → Correct
```

---

# 14. Comparing Stacking With Individual Models

Our final results were:

| Model               | Test Accuracy |
| ------------------- | ------------: |
| Logistic Regression |        96.67% |
| Decision Tree       |        93.33% |
| KNN                 |   **100.00%** |
| Stacking            |   **100.00%** |

So:

```text
KNN       → 100%
Stacking  → 100%
```

---

# 15. Did Stacking Improve Our Result?

**No, not in this experiment.**

KNN already achieved:

```text
100%
```

and stacking also achieved:

```text
100%
```

Therefore:

```text
KNN
 ↓
100%

Stacking
 ↓
100%
```

Stacking **matched** KNN but did not exceed it.

---

# 16. Does Stacking Always Improve Accuracy?

**No.**

This is very important.

Stacking is not guaranteed to perform better than every individual model.

For example:

```text
Model A → 95%
Model B → 94%
Model C → 93%
Stacking → 96%
```

Here stacking improved performance.

But we could also get:

```text
Model A → 100%
Model B → 95%
Model C → 94%
Stacking → 100%
```

Here stacking simply matches the best model.

Or even:

```text
Model A → 95%
Model B → 94%
Model C → 93%
Stacking → 92%
```

Stacking can sometimes perform worse.

The result depends on the dataset, models, and configuration.

---

# 17. Model Comparison vs Stacking

These two concepts are different.

## Model Comparison

We train different models and compare their performance.

```text
                 Dataset
                    ↓
       ┌────────────┼────────────┐
       ↓            ↓            ↓
      LR           Tree          KNN
       ↓            ↓            ↓
     Score        Score         Score
```

Question:

> **Which model performs better?**

---

## Stacking

We actually combine the models.

```text
                 Dataset
                    ↓
       ┌────────────┼────────────┐
       ↓            ↓            ↓
      LR           Tree          KNN
       ↓            ↓            ↓
       └────────────┼────────────┘
                    ↓
               Meta-model
                    ↓
             Final Prediction
```

Question:

> **Can we combine their predictions to make a final prediction?**

---

# 18. Stacking vs Voting

These are both ensemble methods, but the combination works differently.

### Voting

Models vote:

```text
LR       → Class 0
Tree     → Class 0
KNN      → Class 1

Majority → Class 0
```

The combination follows a voting rule.

### Stacking

Models provide predictions to another model:

```text
LR       ──┐
Tree     ──┼──→ Meta-model → Final Prediction
KNN      ──┘
```

The meta-model **learns how to combine the predictions**.

---

# 19. Complete Stacking Workflow

```text
                    Dataset
                       ↓
                Train / Test Split
                       ↓
                  Training Data
                       ↓
          ┌────────────┼────────────┐
          ↓            ↓            ↓
         KNN          Tree          LR
          ↓            ↓            ↓
          └────────────┼────────────┘
                       ↓
               Base Predictions
                       ↓
                  Meta-model
                       ↓
               Final Prediction
                       ↓
                    x_test
                       ↓
                 Final Accuracy
```

---

# ⭐ Important Points to Remember

### Stacking

> **An ensemble technique that combines multiple models using another model.**

### Base Models

> **Models that make the first-level predictions.**

```text
KNN
Decision Tree
Logistic Regression
```

### Meta-Model

> **A model that learns how to combine the predictions of the base models.**

### Meta-model doesn't have to be a base model

```text
Base → KNN + Tree + LR
Meta → Random Forest
```

is valid.

### What goes to the meta-model?

> **Predictions from the base models** — often prediction probabilities for classification.

### Our result

```text
KNN       → 100%
Stacking  → 100%
```

So stacking **matched the best model but didn't improve on it** in our Iris experiment.

---

# 🧠 The One Thing to Memorize

```text
                 Original Data
                       ↓
              Multiple Base Models
                       ↓
                Their Predictions
                       ↓
                  Meta-Model
                       ↓
                Final Prediction
```

> **Base models give their predictions; the meta-model learns how to combine those predictions into the final prediction.**
